# 06 — Protocol Expert Skill

`src/protocol_skill.py` — the `ProtocolSession` class.

## Session lifecycle

```
ProtocolSession.create(protocol, researcher_name, objective)
  ├── load_protocol()                  # download .docx + companion doc
  ├── build_system_prompt()            # protocol + companion injected
  └── session active
        ├── handle_message(text/image) → send to Claude with full context
        ├── handle_deviation(desc)     → log + impact assessment
        ├── handle_refine(finding)     → draft note → append to companion doc
        └── end_session()
              ├── _generate_summary() → Claude structured report
              ├── create_session_doc() + append_doc_text()
              └── append_sheet_row(Lab Journal)
```

In [ ]:
import sys, asyncio
sys.path.insert(0, '..')

from src.protocol_skill import ProtocolSession
from src.google_client import list_protocols
from src.config import get_researcher_name
print('Imports OK')

## 1. Create a ProtocolSession

In [ ]:
protocols = asyncio.run(list_protocols())
if not protocols:
    print('No protocols found. Upload a .docx to the Protocols folder.')
else:
    p = protocols[0]
    session = asyncio.run(ProtocolSession.create(
        protocol=p,
        researcher_name='Dr. Test',
        objective='Verify buffer volumes for standard lysis protocol',
    ))
    print(f'Session created!')
    print(f'  Protocol:      {session.protocol_name}')
    print(f'  Version:       {session.protocol_version}')
    print(f'  Companion doc: {session.companion_doc_id}')
    print(f'  System prompt: {len(session.system_prompt)} chars')

## 2. Send a message through the Protocol Expert

In [ ]:
if protocols:
    reply = asyncio.run(session.handle_message(
        'What is the incubation temperature and duration for this protocol?'
    ))
    print(reply)

## 3. Buffer preparation request

In [ ]:
if protocols:
    reply = asyncio.run(session.handle_message(
        'Buffer preparation: lysis buffer. '
        'Find the recipe in the protocol and ask me for the target final volume.'
    ))
    print(reply)

## 4. Log a deviation

In [ ]:
if protocols:
    reply = asyncio.run(session.handle_deviation(
        'Used 0.5% Triton X-100 instead of 1% — cell viability was a concern.'
    ))
    print(reply)

## 5. /refine — knowledge note

⚠️ If a companion doc exists, this will append a real entry to it.

In [ ]:
# Uncomment to test /refine (writes to companion doc if one exists):
# reply = asyncio.run(session.handle_refine(
#     '0.5% Triton X-100 works as well as 1% for HEK293 cells with fewer aggregates.'
# ))
# print(reply)
print('Uncomment above to test /refine.')

## 6. end_session()

⚠️ This creates a real Google Doc in Session Reports and adds a row to Lab Journal.

In [ ]:
# Uncomment to test end_session() (creates Drive doc + Sheets row):
# summary, doc_url = asyncio.run(session.end_session())
# print(f'Session report: {doc_url}')
# print(f'\nSummary:\n{summary}')
print('Uncomment above to test end_session().')